# Question 1

## How a GEMM code written in Triton gets converted to Nvidia PTX code using MLIR?

Show code fragements after each lowering for a matrix
multiplication of two matrices A and B of size 1024x1024.

In [1]:
!nvidia-smi

Wed Oct 21 17:17:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q triton

In [3]:

import torch
import triton
import triton.language as tl

@triton.jit
def matmul_kernel(
    A, B, C,
    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,
    stride_am: tl.constexpr,
    stride_ak: tl.constexpr,
    stride_bk: tl.constexpr,
    stride_bn: tl.constexpr,
    stride_cm: tl.constexpr,
    stride_cn: tl.constexpr,
    BM: tl.constexpr = 32,
    BN: tl.constexpr = 64,
    BK: tl.constexpr = 32,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    rows = pid_m * BM + tl.arange(0, BM)
    cols = pid_n * BN + tl.arange(0, BN)
    ks = tl.arange(0, BK)

    acc = tl.zeros((BM, BN), dtype=tl.float32)

    for k_block in range(tl.cdiv(K, BK)):
        k = k_block * BK + ks

        a = tl.load(
            A + rows[:, None] * stride_am + k[None, :] * stride_ak,
            mask=(rows[:, None] < M) & (k[None, :] < K), other=0.0,
        )

        b = tl.load(
            B + k[:, None] * stride_bk + cols[None, :] * stride_bn,
            mask=(k[:, None] < K) & (cols[None, :] < N), other=0.0,
        )

        acc = tl.dot(a, b, acc)

    tl.store(
        C + rows[:, None] * stride_cm + cols[None, :] * stride_cn,
        acc, mask=(rows[:, None] < M) & (cols[None, :] < N),
    )


In [4]:

M, K, N = 1024, 1024, 1024

torch.manual_seed(42)
A = torch.randn((M, K), device="cuda", dtype=torch.float32)
B = torch.randn((K, N), device="cuda", dtype=torch.float32)
C = torch.empty((M, N), device="cuda", dtype=torch.float32)

grid = (triton.cdiv(M, 32), triton.cdiv(N, 64))

matmul_kernel[grid](
    A, B, C,
    M, N, K,
    A.stride(0), A.stride(1),
    B.stride(0), B.stride(1),
    C.stride(0), C.stride(1),
    num_warps=4,
)

reference = torch.matmul(A, B)

print("Output shape:", C.shape)
print("Maximum absolute error:",
      (C - reference).abs().max().item())

torch.testing.assert_close(C, reference, rtol=1e-3, atol=1e-3)
print("Correctness test passed!")


Output shape: torch.Size([1024, 1024])
Maximum absolute error: 0.00020599365234375
Correctness test passed!


In [5]:

compiled = matmul_kernel.warmup(
    A, B, C,
    M, N, K,
    A.stride(0), A.stride(1),
    B.stride(0), B.stride(1),
    C.stride(0), C.stride(1),
    grid=grid,
    num_warps=4,
)

print("Available compiler stages:")
print(list(compiled.asm.keys()))

for stage in ["ttir", "ttgir", "llir", "ptx"]:
    print(f"\n{'=' * 25} {stage.upper()} {'=' * 25}")
    if stage in compiled.asm:
        print(compiled.asm[stage])
    else:
        print("This stage is not exposed in this Triton version.")


Available compiler stages:
['source', 'ttir', 'ttgir', 'llir', 'ptx', 'cubin']

========================= TTIR =========================
#loc = loc("/tmp/ipykernel_1818/54025802.py":6:0)
#loc43 = loc("A"(#loc))
#loc44 = loc("B"(#loc))
#loc45 = loc("C"(#loc))
module {
  tt.func public @matmul_kernel(%A: !tt.ptr<f32> {tt.divisibility = 16 : i32} loc("A"(#loc)), %B: !tt.ptr<f32> {tt.divisibility = 16 : i32} loc("B"(#loc)), %C: !tt.ptr<f32> {tt.divisibility = 16 : i32} loc("C"(#loc))) attributes {noinline = false} {
    %c1_i32 = arith.constant 1 : i32 loc(#loc1)
    %c0_i32 = arith.constant 0 : i32 loc(#loc1)
    %cst = arith.constant dense<0.000000e+00> : tensor<32x64xf32> loc(#loc2)
    %cst_0 = arith.constant dense<1024> : tensor<1x64xi32> loc(#loc2)
    %cst_1 = arith.constant dense<0.000000e+00> : tensor<32x32xf32> loc(#loc2)
    %cst_2 = arith.constant dense<1024> : tensor<1x32xi32> loc(#loc2)
    %cst_3 = arith.constant dense<1024> : tensor<32x1xi32> loc(#loc2)
    %c64_i32 = arith